## Finding all adverbials from a database

The aim of this notebook is to find all adverbials from the Estonian Reference corpus. This is needed to annotate them with semantic class using both rule based methods and LLMs. The code extracts data from Katrin Tsepelina's database [v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db](https://github.com/estnltk/syntax_experiments/tree/verb_templates/workflows/001_verb_transactions/v33) with data extracted from the Estonian Reference corpus.

This code creates a table for GPT labelled data based on the tag.

In [1]:
#imports
import sqlite3
import pandas as pd
from tqdm import tqdm

In [2]:
# database file path
DB_FILE = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310_gpt_labelled.db"

GPT_TABLE = f"gpt_labelled_all"

# column for final new tag
TAG_COL = "gpt_tags"

In [3]:
def column_exists(cursor, table, column):
    cursor.execute(f"PRAGMA table_info({table})")
    return any(row[1] == column for row in cursor.fetchall())

In [4]:
# connecting with database
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

### Create new table

In [5]:
eltn50 = f"../data/datasets_500_all_results/finished/ELT_n50_500_batch_tagged_final.csv"
eltn80 = f"../data/datasets_500_all_results/finished/ELT_n80_500_batch_tagged_final.csv"
an50 = f"../data/datasets_500_all_results/finished/A_n50_500_batch_tagged_final.csv"
an80 = f"../data/datasets_500_all_results/finished/A_n80_500_batch_tagged_final.csv"
sn50 = f"../data/datasets_500_all_results/finished/S_n50_500_batch_tagged_final.csv"
sn80 = f"../data/datasets_500_all_results/finished/S_n80_500_batch_tagged_final.csv"

In [6]:
eltn50df = pd.read_csv(eltn50, sep=",", encoding="utf-8")
eltn80df = pd.read_csv(eltn80, sep=",", encoding="utf-8")
an50df = pd.read_csv(an50, sep=",", encoding="utf-8")
an80df = pd.read_csv(an80, sep=",", encoding="utf-8")
sn50df = pd.read_csv(sn50, sep=",", encoding="utf-8")
sn80df = pd.read_csv(sn80, sep=",", encoding="utf-8")

In [26]:
eltn50df["initial_cat"] = "elt_n50"
eltn80df["initial_cat"] = "elt_n80"
an50df["initial_cat"] = "a_n50"
an80df["initial_cat"] = "a_n80"
sn50df["initial_cat"] = "s_n50"
sn80df["initial_cat"] = "s_n80"

In [27]:
dfgpt = pd.concat([eltn50df, eltn80df, an50df, an80df, sn50df, sn80df], ignore_index=True)

dfgpt["id"] = dfgpt.index
dfgpt["verb_compound"] = dfgpt["verb_compound"].fillna("")

In [28]:
len(dfgpt)

515893

In [29]:
dfgpt.head(3)

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,...,A,E,T,L1,L2,L,S,tag,initial_cat,id
0,709965,1130306,13,tõmbama,,all,Vilsandi,Vilsandile,Kuigi Norrköpingi lähedal saarel on kalurist i...,|L|AL|EL|LS|LT|ALT|ELT|LST|,...,no,no,no,yes,no,yes,no,L,elt_n50,0
1,265424,419095,2,rõõmustama,,el,mis,millest,"... millest kõigest , niipea kui see meie arms...",NaN,...,no,no,no,no,no,no,no,NaN,elt_n50,1
2,1585112,2521697,10,lahutama,,el,allkiri,allkirjadest,"Veel eelmise linnavalitsuse ajal , aasta tagas...",NaN,...,no,no,no,yes,no,yes,no,L,elt_n50,2


In [30]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

dfgpt.to_sql(GPT_TABLE, conn, if_exists="replace", index=False)

conn.close()

### Create tags column based on gpt predicted tag

For tag combinations:

- EVENT - E
- TIME - T
- LOC - L
- LOCEVENT - EL
- LOCEVENTTIME - ELT
- ALIVE - A
- STATE - S

In [31]:

combinations = { 
    "T": ["T","LT","ET","ELT","AT","ALT","AET","LST","AST","ST"], 
    "L": ["L","LT","EL","ELT","LS","AL","ALT","LST"], 
    "E": ["E","EL","ET","ELT","AE","AET",], 
    "A": ["A","AT","AL","AE","AS","ALT","AET","AST"], 
    "S": ["S","LS","AS","LST","AST","ST",], 
}


def get_combos(letters, limit):
    letters_set = set(letters)
    result = []
    for letter in letters:
        for combo in combinations[letter]:
            # combo has the right amount of tags and not other tags
            if len(combo) > limit and letters_set.issubset(combo):
                result.append(combo)

    return result


def get_tags(gpt_tag):

    sources = []
    # Info about what tags are present
    if gpt_tag is not None:
        sources.append(gpt_tag)

    if not sources:
        return []

    letters = set(sources)

    if len(letters) == 1: # only one tag is present, get 1,2,3 letter combinations
        tags = combinations[next(iter(letters))]
    else: # more than one tag, get 2+3 or just 3 letter combinations
        limit = 1 if len(letters) == 2 else 2
        tags = list(set(get_combos(letters, limit)))

    # alphabetically and order by 1, letter tags, 2-letter tags, 3-letter tags
    tags.sort(key=lambda w: (len(w), w))
    return tags

In [32]:
# connecting with database
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

In [33]:
# get necessary columns from obl table
query = f"SELECT id, tag FROM {GPT_TABLE}"
obl_tags = pd.read_sql(query, conn)
#obl_tags # 375,810 rows

In [34]:
# Get tags for all rows

tags_col = []

for i in tqdm(range(len(obl_tags))):
    idx = obl_tags.iloc[i]["id"]
    gpt = obl_tags.iloc[i]["tag"]

    tag_list = get_tags(gpt)
    if len(tag_list) != 0:
        tag_str = "|" + "|".join(tag_list) + "|"
    else:
        tag_str = ""
        
    tags_col.append((idx, tag_str))

tags_col_clean = [(int(i), t) for i, t in tags_col]
assert len(tags_col_clean) == len(obl_tags)

100%|████████████████████████████████| 515893/515893 [00:31<00:00, 16479.12it/s]


In [35]:

# New column if it doesn't exist
if not column_exists(cursor, GPT_TABLE, TAG_COL):
    cursor.execute("ALTER TABLE " + GPT_TABLE +  f" ADD COLUMN {TAG_COL} TEXT")

# Create a temporary table
cursor.execute("CREATE TEMP TABLE temp_tags (id INT PRIMARY KEY, tags TEXT)")

# Insert all values into the temp table
cursor.executemany("INSERT INTO temp_tags (id, tags) VALUES (?, ?)", tags_col_clean)

# Perform a fast join-based update
query = f"""
    UPDATE {GPT_TABLE}
    SET {TAG_COL} = (
        SELECT tags
        FROM temp_tags
        WHERE temp_tags.id = {GPT_TABLE}.id
        LIMIT 1
     )
    WHERE EXISTS (
        SELECT 1
        FROM temp_tags
        WHERE temp_tags.id = {GPT_TABLE}.id
    )
"""

cursor.execute(query)

conn.commit()
conn.close()

## Extra column tag2 

Contains only ELT, A, S or ""

In [55]:
# change if needed
def new_class(row):
    if row["tag"] == "E" or row["tag"] == "L" or row["tag"] == "T":
        return "ELT"
    if row["tag"] == "A":
        return "A"
    if row["tag"] == "S":
        return "S"
    else:
        return ""

In [56]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

query = f"SELECT * FROM {GPT_TABLE}"
df = pd.read_sql(query, conn)

conn.close()

In [57]:
df["tag2"] = df.apply(new_class, axis=1)

In [58]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,...,T,L1,L2,L,S,tag,initial_cat,id,gpt_tags,tag2
0,709965,1130306,13,tõmbama,,all,Vilsandi,Vilsandile,Kuigi Norrköpingi lähedal saarel on kalurist i...,|L|AL|EL|LS|LT|ALT|ELT|LST|,...,no,yes,no,yes,no,L,elt_n50,0,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
1,265424,419095,2,rõõmustama,,el,mis,millest,"... millest kõigest , niipea kui see meie arms...",None,...,no,no,no,no,no,None,elt_n50,1,,
2,1585112,2521697,10,lahutama,,el,allkiri,allkirjadest,"Veel eelmise linnavalitsuse ajal , aasta tagas...",None,...,no,yes,no,yes,no,L,elt_n50,2,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
3,6601769,10623242,13,lahutama,,el,neid,neiust,""" Nõidkütt "" on lugu jahiõnne kaotanud laskuri...",|A|AE|AL|AS|AT|AET|ALT|AST|,...,no,no,no,no,no,A,elt_n50,3,|A|AE|AL|AS|AT|AET|ALT|AST|,A
4,15317061,23915903,7,viskama,,all,kaptenisild,kaptenisillale,Näiteks kui sa viskad mingi suitsupommi kapten...,|L|AL|EL|LS|LT|ALT|ELT|LST|,...,no,yes,no,yes,no,L,elt_n50,4,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515888,10711639,17164924,6,surema,,adit,valdus,valdusse,Frank ja Vinnie saavad oma valdusse Surnud Koe...,None,...,no,yes,no,yes,no,L,s_n80,515888,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
515889,5235186,8398817,3,haigestuma,,ill,valgusallergia,valgusallergiasse,Hannelore haigestus valgusallergiasse 1993. aa...,None,...,no,no,no,no,no,None,s_n80,515889,,
515890,8758718,14058979,14,surema,,adit,külm,külma,"Enne seda , kui emad õppisid varrastel kuduma ...",None,...,no,no,no,no,no,None,s_n80,515890,,
515891,11717337,18752192,1,surema,,adit,ving,Vingu,Vingu ja põletushaavadesse on aga surnud 64-aa...,None,...,no,no,no,no,yes,S,s_n80,515891,|S|AS|LS|ST|AST|LST|,S


In [59]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

df.to_sql(GPT_TABLE, conn, if_exists="replace", index=False)

conn.close()

## DUPLICATE check

## Puhastamata tabelist duplikaadid (lihtsalt infoks)

In [60]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

query = f"SELECT * FROM {GPT_TABLE}"
res = pd.read_sql(query, conn)

conn.close()

In [61]:
key_cols = ["sentence_id", "head_id", "head_loc", "verb", "verb_compound", "morph_case", "lemma", "form", "sentence"]

duplicates = (
    res[res.duplicated(
        subset=key_cols,
        keep=False
    )]
    .sort_values(key_cols)
)

In [62]:
duplicates

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,...,T,L1,L2,L,S,tag,initial_cat,id,gpt_tags,tag2
53558,490,881,17,kujutama,,el,ise,endast,"Tema värskeim film , Analyze That ( “ Gangster...",None,...,no,no,no,no,no,None,elt_n50,53558,,
448080,490,881,17,kujutama,,el,ise,endast,"Tema värskeim film , Analyze That ( “ Gangster...",None,...,no,no,no,no,no,None,a_n50,448080,,
83538,490,881,22,kujutama,,all,film,filmile,"Tema värskeim film , Analyze That ( “ Gangster...",None,...,no,no,no,no,no,None,elt_n50,83538,,
388467,490,881,22,kujutama,,all,film,filmile,"Tema värskeim film , Analyze That ( “ Gangster...",None,...,no,yes,no,yes,no,L,a_n50,388467,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
89241,541,975,19,juhatama,,all,see,neile,"Tema uus armastatu , DJ Priit Kuusik imestab s...",None,...,no,no,no,no,no,A,elt_n50,89241,|A|AE|AL|AS|AT|AET|ALT|AST|,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
409175,21404246,30069493,7,istuma,,all,põlwe,põlwele,romeo: crista tule istu mu põlwele ja räägi ke...,None,...,no,yes,no,yes,no,L,a_n50,409175,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
122544,21412671,30073242,7,hoidma,eemale,el,seltskond,seltskonnast,xav: no siis hoia minu seltskonnast eemale,None,...,no,yes,no,yes,no,L,elt_n50,122544,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
413558,21412671,30073242,7,hoidma,eemale,el,seltskond,seltskonnast,xav: no siis hoia minu seltskonnast eemale,None,...,no,yes,no,yes,no,L,a_n50,413558,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
53455,21418307,30076043,10,olema,kohal,el,kedagi,kedagist,Silent_Trigger: raadiost ei tea miskit - pole ...,None,...,no,no,no,no,no,A,elt_n50,53455,|A|AE|AL|AS|AT|AET|ALT|AST|,A


## Remove duplicates based on tag2

In [63]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

query = f"SELECT * FROM {GPT_TABLE}"
res = pd.read_sql(query, conn)

conn.close()

In [64]:
key_cols = [
    "sentence_id", "head_id", "head_loc", "verb",
    "verb_compound", "morph_case", "lemma", "form", "sentence"
]

def normalize_tag(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def resolve_group(group):
    tags = group["tag2"].apply(normalize_tag)

    # rows with a non-empty tag
    nonempty = group[tags != ""]

    # all empty -> keep one
    if len(nonempty) == 0:
        return group.iloc[[0]]

    # only one non-empty -> keep that one
    if len(nonempty) == 1:
        return nonempty

    # multiple non-empty values
    unique_tags = set(nonempty["tag2"].apply(normalize_tag))

    # all non-empty tags are identical -> keep one
    if len(unique_tags) == 1:
        return nonempty.iloc[[0]]

    # different non-empty tags -> keep one row per distinct tag
    return (
        nonempty.assign(_tag=nonempty["tag2"].apply(normalize_tag))
        .drop_duplicates("_tag")
        .drop(columns="_tag")
    )

result = (
    res.groupby(key_cols, dropna=False, group_keys=False)
       .apply(resolve_group)
       .reset_index(drop=True)
)

In [65]:
result

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,...,T,L1,L2,L,S,tag,initial_cat,id,gpt_tags,tag2
0,37,51,8,kutsuma,,adit,elu,ellu,"Pole välistatud , et me pundi kunagi ellu kuts...",None,...,no,yes,no,yes,no,L,elt_n80,164368,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
1,54,85,9,kirjutama,,all,bänd,bändidele,Olen viimasel ajal juhutöödest loobunud ja kir...,|A|AE|AL|AS|AT|AET|ALT|AST|,...,no,no,no,no,no,A,a_n80,485607,|A|AE|AL|AS|AT|AET|ALT|AST|,A
2,81,131,8,tiirutama,,ad,tuur,tuuridel,Saime tuhandeid kirju ja meie ümber tiirutasid...,None,...,no,no,no,no,no,E,elt_n80,131533,|E|AE|EL|ET|AET|ELT|,ELT
3,84,142,14,helistama,,el,hommik,hommikust,"Mõned tüdrukud muutusid lausa tüütuks , uurisi...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,...,yes,no,no,no,no,T,elt_n80,328034,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT
4,92,157,2,täitma,,ad,aasta,aastal,"Sel aastal täitsime oma missiooni , mille eesm...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,...,yes,no,no,no,no,T,elt_n80,140113,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432097,21413464,30073669,6,kaduma,,el,aur,aurudest,+Lenka: mul juba aurudest kadus lendab,None,...,no,yes,no,yes,no,L,elt_n80,202877,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
432098,21414898,30074249,7,saama,valmis,ad,oma,omal,Rex: ma sain isegi jutuka omal valmis,None,...,no,no,no,no,no,None,elt_n80,362774,,
432099,21415454,30074507,6,laskma,,adit,datanet,datanetti,+Knight17_away: metscat lase datanetti üless :),None,...,no,yes,no,yes,no,L,elt_n80,339718,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
432100,21415689,30074650,4,saama,,ill,kasiioo,kasiioosse,Rex: nooremad kasiioosse ei saa,None,...,no,yes,no,yes,no,L,elt_n80,288450,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT


In [70]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

result.to_sql("gpt_labelled_clean", conn, if_exists="replace", index=False)

conn.close()

In [66]:
key_cols = ["sentence_id", "head_id", "head_loc", "verb", "verb_compound", "morph_case", "lemma", "form", "sentence"]

duplicates2 = (
    result[result.duplicated(
        subset=key_cols,
        keep=False
    )]
    .sort_values(key_cols)
)

In [67]:
duplicates2

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,...,T,L1,L2,L,S,tag,initial_cat,id,gpt_tags,tag2
51,845,1538,22,langema,,adit,minestus,minestusse,"Kuuldes , et läheb peagi mehest lahku , abiell...",|S|AS|LS|ST|AST|LST|,...,no,no,no,no,yes,S,elt_n50,5431,|S|AS|LS|ST|AST|LST|,S
52,845,1538,22,langema,,adit,minestus,minestusse,"Kuuldes , et läheb peagi mehest lahku , abiell...",|S|AS|LS|ST|AST|LST|,...,no,no,no,no,no,E,s_n50,514956,|E|AE|EL|ET|AET|ELT|,ELT
462,7619,13038,20,jääma,ilma,el,jalg,jalast,"Heatheri esimene autobiograafia , mis ilmus 19...",None,...,no,yes,no,yes,no,L,elt_n50,76010,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
463,7619,13038,20,jääma,ilma,el,jalg,jalast,"Heatheri esimene autobiograafia , mis ilmus 19...",None,...,no,no,no,no,yes,S,a_n50,467322,|S|AS|LS|ST|AST|LST|,S
471,7760,13304,4,uskuma,,ill,jumal,jumalasse,"Ma ei usu jumalasse , iseennast tuleb uskuda .",None,...,no,no,no,no,no,A,elt_n50,126869,|A|AE|AL|AS|AT|AET|ALT|AST|,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
431825,21230462,29985654,6,kartma,,el,pimedus,pimedusest,High-Q: mis te sellest pimedusest kardate,None,...,no,no,no,no,yes,S,a_n50,377345,|S|AS|LS|ST|AST|LST|,S
431908,21288530,30012817,9,vaatama,,all,k6igi,k6igile,def_siinsealsiin: liis_musi vaata et siis kohe...,None,...,no,yes,no,yes,no,L,elt_n50,98566,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
431909,21288530,30012817,9,vaatama,,all,k6igi,k6igile,def_siinsealsiin: liis_musi vaata et siis kohe...,None,...,no,no,no,no,no,A,a_n50,464978,|A|AE|AL|AS|AT|AET|ALT|AST|,A
432004,21363576,30048148,6,aitama,,all,pyro,pyrole,+dim: siis kui pyrole aitasime accessi anda,None,...,no,yes,no,yes,no,L,elt_n50,66212,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT


In [69]:
# kui palju on lauseid, mille puhul gpt andis erinevaid vastuseid sest lause dolid eri andmestikes
len(duplicates2["sentence_id"].unique())

3090

## Remove duplicates based on gpt_tags

In [71]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

query = f"SELECT * FROM {GPT_TABLE}"
res = pd.read_sql(query, conn)

conn.close()

In [72]:
key_cols = [
    "sentence_id", "head_id", "head_loc", "verb",
    "verb_compound", "morph_case", "lemma", "form", "sentence"
]

def normalize_tag(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def resolve_group(group):
    tags = group["gpt_tags"].apply(normalize_tag)

    # rows with a non-empty tag
    nonempty = group[tags != ""]

    # all empty -> keep one
    if len(nonempty) == 0:
        return group.iloc[[0]]

    # only one non-empty -> keep that one
    if len(nonempty) == 1:
        return nonempty

    # multiple non-empty values
    unique_tags = set(nonempty["gpt_tags"].apply(normalize_tag))

    # all non-empty tags are identical -> keep one
    if len(unique_tags) == 1:
        return nonempty.iloc[[0]]

    # different non-empty tags -> keep one row per distinct tag
    return (
        nonempty.assign(_tag=nonempty["gpt_tags"].apply(normalize_tag))
        .drop_duplicates("_tag")
        .drop(columns="_tag")
    )

result = (
    res.groupby(key_cols, dropna=False, group_keys=False)
       .apply(resolve_group)
       .reset_index(drop=True)
)

In [73]:
result

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,...,T,L1,L2,L,S,tag,initial_cat,id,gpt_tags,tag2
0,37,51,8,kutsuma,,adit,elu,ellu,"Pole välistatud , et me pundi kunagi ellu kuts...",None,...,no,yes,no,yes,no,L,elt_n80,164368,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
1,54,85,9,kirjutama,,all,bänd,bändidele,Olen viimasel ajal juhutöödest loobunud ja kir...,|A|AE|AL|AS|AT|AET|ALT|AST|,...,no,no,no,no,no,A,a_n80,485607,|A|AE|AL|AS|AT|AET|ALT|AST|,A
2,81,131,8,tiirutama,,ad,tuur,tuuridel,Saime tuhandeid kirju ja meie ümber tiirutasid...,None,...,no,no,no,no,no,E,elt_n80,131533,|E|AE|EL|ET|AET|ELT|,ELT
3,84,142,14,helistama,,el,hommik,hommikust,"Mõned tüdrukud muutusid lausa tüütuks , uurisi...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,...,yes,no,no,no,no,T,elt_n80,328034,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT
4,92,157,2,täitma,,ad,aasta,aastal,"Sel aastal täitsime oma missiooni , mille eesm...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,...,yes,no,no,no,no,T,elt_n80,140113,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434365,21413464,30073669,6,kaduma,,el,aur,aurudest,+Lenka: mul juba aurudest kadus lendab,None,...,no,yes,no,yes,no,L,elt_n80,202877,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
434366,21414898,30074249,7,saama,valmis,ad,oma,omal,Rex: ma sain isegi jutuka omal valmis,None,...,no,no,no,no,no,None,elt_n80,362774,,
434367,21415454,30074507,6,laskma,,adit,datanet,datanetti,+Knight17_away: metscat lase datanetti üless :),None,...,no,yes,no,yes,no,L,elt_n80,339718,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
434368,21415689,30074650,4,saama,,ill,kasiioo,kasiioosse,Rex: nooremad kasiioosse ei saa,None,...,no,yes,no,yes,no,L,elt_n80,288450,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT


In [74]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

result.to_sql("gpt_labelled_clean2", conn, if_exists="replace", index=False)

conn.close()

In [75]:
key_cols = ["sentence_id", "head_id", "head_loc", "verb", "verb_compound", "morph_case", "lemma", "form", "sentence"]

duplicates3 = (
    result[result.duplicated(
        subset=key_cols,
        keep=False
    )]
    .sort_values(key_cols)
)

In [76]:
duplicates3

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,...,T,L1,L2,L,S,tag,initial_cat,id,gpt_tags,tag2
51,845,1538,22,langema,,adit,minestus,minestusse,"Kuuldes , et läheb peagi mehest lahku , abiell...",|S|AS|LS|ST|AST|LST|,...,no,no,no,no,yes,S,elt_n50,5431,|S|AS|LS|ST|AST|LST|,S
52,845,1538,22,langema,,adit,minestus,minestusse,"Kuuldes , et läheb peagi mehest lahku , abiell...",|S|AS|LS|ST|AST|LST|,...,no,no,no,no,no,E,s_n50,514956,|E|AE|EL|ET|AET|ELT|,ELT
65,1006,1862,18,seisma,ees,ad,lähikuu,lähikuudel,Näitleja ja lavastaja Mikk Mikiveril ( 65 ) ja...,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,...,no,no,no,no,no,E,elt_n50,100745,|E|AE|EL|ET|AET|ELT|,ELT
66,1006,1862,18,seisma,ees,ad,lähikuu,lähikuudel,Näitleja ja lavastaja Mikk Mikiveril ( 65 ) ja...,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,...,yes,no,no,no,no,T,a_n50,478901,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT
273,4776,8281,7,mõtlema,,el,jalgpall,jalgpallist,Aga mõtleb ta ka siis ikka jalgpallist .,None,...,no,no,no,no,no,E,elt_n50,10683,|E|AE|EL|ET|AET|ELT|,ELT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434093,21230462,29985654,6,kartma,,el,pimedus,pimedusest,High-Q: mis te sellest pimedusest kardate,None,...,no,no,no,no,yes,S,a_n50,377345,|S|AS|LS|ST|AST|LST|,S
434176,21288530,30012817,9,vaatama,,all,k6igi,k6igile,def_siinsealsiin: liis_musi vaata et siis kohe...,None,...,no,yes,no,yes,no,L,elt_n50,98566,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
434177,21288530,30012817,9,vaatama,,all,k6igi,k6igile,def_siinsealsiin: liis_musi vaata et siis kohe...,None,...,no,no,no,no,no,A,a_n50,464978,|A|AE|AL|AS|AT|AET|ALT|AST|,A
434272,21363576,30048148,6,aitama,,all,pyro,pyrole,+dim: siis kui pyrole aitasime accessi anda,None,...,no,yes,no,yes,no,L,elt_n50,66212,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT


In [77]:
# kui palju oli duplikaate (arvuline kontroll)
(len(dfgpt) - len(result))*2 + len(duplicates3)

173768

In [78]:
(len(dfgpt) - len(result))*2 + len(duplicates3) == len(duplicates)

True

## Add transaction_head table if missing

In [46]:
DB2 = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310_test.db"
               
conn = sqlite3.connect(DB2)

query = f"SELECT * FROM transaction_head"
res = pd.read_sql(query, conn)

conn.close()



conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

res.to_sql("transaction_head", conn, if_exists="replace", index=False)

conn.commit()
conn.close()

